# 02 — TypiClust Baseline Evaluation

Fully supervised evaluation of TPC_RP vs Random on CIFAR-10.

**Setup:** B=10/round, 5 rounds, 3 reps, classifier trained from scratch each round.

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
else:
    sys.path.insert(0, os.getcwd())

import numpy as np
import torch
from torchvision import datasets
from src.active_learning import run_repeated_experiment
from src.augmentations import get_classifier_train_transform, get_classifier_test_transform
from src.utils import get_device, set_seed, MODELS_DIR, RESULTS_DIR, DATA_DIR, save_results
set_seed(42)
device = get_device()

## 1. Load Features & Data

In [ ]:
features = np.load(str(MODELS_DIR / 'simclr_features.npy'))
print(f'Features: {features.shape}')

train_dataset = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True, transform=get_classifier_train_transform())
test_dataset = datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True, transform=get_classifier_test_transform())

## 2. TypiClust (Euclidean — Paper's Method)

In [ ]:
results_tc = run_repeated_experiment(
    features=features, train_dataset=train_dataset, test_dataset=test_dataset,
    strategy='typiclust', budget_per_round=10, n_rounds=5, n_reps=5,
    classifier_epochs=200, device=device, typicality_fn='euclidean', base_seed=42,
)
save_results(results_tc, 'typiclust_euclidean_b10.json')
for b, a, s in zip(results_tc['cumulative_budget'], results_tc['mean_accuracy'], results_tc['se_accuracy']):
    print(f'  Budget {b}: {a:.2f}% +/- {s:.2f}%')

## 3. Random Baseline

In [ ]:
results_rd = run_repeated_experiment(
    features=features, train_dataset=train_dataset, test_dataset=test_dataset,
    strategy='random', budget_per_round=10, n_rounds=5, n_reps=3,
    classifier_epochs=100, device=device, base_seed=42,
)
save_results(results_rd, 'random_b10.json')
for b, a, s in zip(results_rd['cumulative_budget'], results_rd['mean_accuracy'], results_rd['se_accuracy']):
    print(f'  Budget {b}: {a:.2f}% +/- {s:.2f}%')

## 4. Plot Results

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
budgets = results_tc['cumulative_budget']

for res, label, color, marker, ls in [
    (results_tc, 'TypiClust (Euclidean)', 'tab:blue', 'o', '-'),
    (results_rd, 'Random', 'tab:gray', 's', '--'),
]:
    m = np.array(res['mean_accuracy']); s = np.array(res['se_accuracy'])
    ax.plot(budgets, m, f'{marker}{ls}', label=label, color=color)
    ax.fill_between(budgets, m-s, m+s, alpha=0.2, color=color)

ax.set_xlabel('Cumulative Budget'); ax.set_ylabel('Test Accuracy (%)')
ax.set_title('CIFAR-10: TypiClust vs Random (Fully Supervised)')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('../figures/al_baseline_results.png', dpi=150); plt.show()